# 03 — Baseline Models: Seasonal Naive, SARIMA and Prophet

Trains the classical forecasting models and evaluates them on 2025 using a rolling forecast origin. Every model is scored the same way, and its predictions are saved in a shared format so the LSTM and the final comparison can use them.

**Evaluation design**
- **Train:** 2021–2024. **Test:** 2025.
- **Rolling forecasts:** every day from 1 Jan to 25 Dec 2025, each model forecasts the next 7 days using only data up to the day before. That's 359 forecasts × 7 days = 2,513 predictions per model.
- A new forecast every day (rather than weekly) means each horizon (1–7 days ahead) covers all seven weekdays equally, so horizon errors aren't mixed up with day-of-week effects.
- **Metrics** (from `src/metrics.py`): MAE, RMSE, MAPE and MASE. MASE below 1 means better than the seasonal-naive forecast.

In [1]:
import sys
import time
import warnings
import numpy as np
import pandas as pd
sys.path.append("..")
from src.config import *
from src.metrics import score

## 1. Load data and forecast origins
Settings (paths, train/test split, horizon, step) all come from `src/config.py`, so every model uses the identical setup.

In [2]:
daily = pd.read_csv(PROCESSED_DATA, index_col="date", parse_dates=True)
y = daily[TARGET]
y_train = y[:TRAIN_END]
origins = forecast_origins()
print(len(origins), origins[0].date(), origins[-1].date())   

359 2025-01-01 2025-12-25


## 2. Shared scoring and saving
Every model's predictions go through this one function: score with the shared metrics, show the error at each horizon, and save to `results/predictions/<model>.csv` in the same column format (`model, origin, date, horizon, y_true, y_pred`).

In [3]:
def evaluate_and_save(preds, name):
    """Score a model's predictions, show error by horizon, and save them."""
    scores = score(preds["y_true"], preds["y_pred"], y_train=y_train, m=SEASONAL_PERIOD)
    print(f"=== {name} ===")
    for k, v in scores.items():
        print(f"  {k:5s} {v:,.3f}")

    by_horizon = (preds.assign(abs_error=(preds["y_true"] - preds["y_pred"]).abs())
                       .groupby("horizon")["abs_error"].mean().round(0))
    print("\n  MAE by horizon (days ahead):")
    print(by_horizon.to_string())

    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    out = PREDICTIONS_DIR / f"{name}.csv"
    preds.to_csv(out, index=False)
    print(f"\n  Saved {len(preds)} rows to {out}")
    return scores

## 3. Seasonal naive ("same day last week")
The benchmark every model has to beat: the forecast for each day is the actual demand on the same weekday one week earlier. That value is always known when the forecast is made, so there's no leakage.

In [4]:
rows = []
for origin in origins:
    for h, d in enumerate(pd.date_range(origin, periods=HORIZON), start=1):
        rows.append({
            "model": "seasonal_naive",
            "origin": origin,
            "date": d,
            "horizon": h,
            "y_true": y[d],
            "y_pred": y[d - pd.Timedelta(days=SEASONAL_PERIOD)],   # same weekday, one week earlier
        })

naive_preds = pd.DataFrame(rows)
naive_scores = evaluate_and_save(naive_preds, "seasonal_naive")

=== seasonal_naive ===
  MAE   1,821.982
  RMSE  2,392.299
  MAPE  6.885
  MASE  1.045

  MAE by horizon (days ahead):
horizon
1    1844.0
2    1852.0
3    1837.0
4    1823.0
5    1812.0
6    1800.0
7    1786.0

  Saved 2513 rows to /Users/jay/Documents/Project 5/uk-grid-demand-forecasting/results/predictions/seasonal_naive.csv


## 4. SARIMA

### 4a. Model selection
`auto_arima` searches for the best model structure once, on 2021–2024 only.
- **Weekly seasonality** (`m = 7`), with weekly differencing **fixed at `D = 1`** rather than left to `auto_arima`'s automatic seasonal test. This was decided from the EDA before seeing any test results: the weekly cycle is strong and the weekly-differenced series is clearly stationary (ADF p < 0.001). In a trial on synthetic data with the same structure, the automatic test skipped weekly differencing and produced a much weaker model.
- **Regular differencing (`d`)** is left to `auto_arima`. The ADF result suggested none would be needed.
- **Calendar inputs:** `is_holiday` and `is_christmas_period`. These are known in advance, so using them for future dates isn't leakage.
- SARIMA supports only **one** seasonal period, so it can't model the yearly cycle directly. This is the key difference from Prophet.

In [5]:
from pmdarima import auto_arima

# Calendar features are known in advance, so they can be used for future dates without leakage
EXOG = ["is_holiday", "is_christmas_period"]
X = daily[EXOG]
X_train = X[:TRAIN_END]

t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarima = auto_arima(
        y_train, X=X_train,
        seasonal=True, m=SEASONAL_PERIOD,
        D=1,                                   # weekly differencing, fixed on purpose (see below)
        stepwise=True,
        max_p=3, max_q=3, max_P=2, max_Q=2,
        information_criterion="aic",
        suppress_warnings=True, error_action="ignore",
        trace=True,                            # prints each model it tries, so you can see progress
    )
print(f"\nauto_arima took {time.time() - t0:.0f}s")
print("Chosen model: order", sarima.order, " seasonal_order", sarima.seasonal_order)

Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,1,1)[7] intercept   : AIC=inf, Time=3.09 sec
 ARIMA(0,0,0)(0,1,0)[7] intercept   : AIC=26338.546, Time=0.03 sec
 ARIMA(1,0,0)(1,1,0)[7] intercept   : AIC=25127.553, Time=0.71 sec
 ARIMA(0,0,1)(0,1,1)[7] intercept   : AIC=25497.062, Time=0.77 sec
 ARIMA(0,0,0)(0,1,0)[7]             : AIC=26337.022, Time=0.17 sec
 ARIMA(1,0,0)(0,1,0)[7] intercept   : AIC=25544.274, Time=0.11 sec
 ARIMA(1,0,0)(2,1,0)[7] intercept   : AIC=25030.467, Time=1.29 sec
 ARIMA(1,0,0)(2,1,1)[7] intercept   : AIC=24949.202, Time=2.65 sec
 ARIMA(1,0,0)(1,1,1)[7] intercept   : AIC=24955.052, Time=1.40 sec
 ARIMA(1,0,0)(2,1,2)[7] intercept   : AIC=25012.265, Time=3.37 sec
 ARIMA(1,0,0)(1,1,2)[7] intercept   : AIC=24958.631, Time=2.62 sec
 ARIMA(0,0,0)(2,1,1)[7] intercept   : AIC=26226.046, Time=1.85 sec
 ARIMA(2,0,0)(2,1,1)[7] intercept   : AIC=24953.854, Time=3.38 sec
 ARIMA(1,0,1)(2,1,1)[7] intercept   : AIC=24997.276, Time=3.08 sec
 ARIMA(0,0,1)(2,1,1)[7] i

### 4b. Rolling forecasts
The model structure stays fixed. Before each forecast, `update()` adds the actual demand observed since the previous origin, then the model predicts the next 7 days. This mirrors how it would run in practice, and is much faster than refitting 359 times.

In [6]:
one_day = pd.Timedelta(days=1)
rows = []
last_seen = TRAIN_END

t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for origin in origins:
        # 1. Feed in the actual values observed since the last origin (the week that just passed)
        new_y = y[last_seen + one_day : origin - one_day]
        if len(new_y) > 0:
            sarima.update(new_y, X=X.loc[new_y.index])
        last_seen = origin - one_day

        # 2. Forecast the next 7 days
        forecast_dates = pd.date_range(origin, periods=HORIZON)
        y_pred = np.asarray(sarima.predict(n_periods=HORIZON, X=X.loc[forecast_dates]))

        for h, (d, p) in enumerate(zip(forecast_dates, y_pred), start=1):
            rows.append({"model": "sarima", "origin": origin, "date": d,
                         "horizon": h, "y_true": y[d], "y_pred": p})

print(f"Rolling forecasts took {time.time() - t0:.0f}s")
sarima_preds = pd.DataFrame(rows)
sarima_scores = evaluate_and_save(sarima_preds, "sarima")

Rolling forecasts took 138s
=== sarima ===
  MAE   1,493.125
  RMSE  1,934.043
  MAPE  5.758
  MASE  0.856

  MAE by horizon (days ahead):
horizon
1     973.0
2    1247.0
3    1376.0
4    1541.0
5    1666.0
6    1784.0
7    1865.0

  Saved 2513 rows to /Users/jay/Documents/Project 5/uk-grid-demand-forecasting/results/predictions/sarima.csv


## 5. Prophet

### 5a. Setup
Prophet models **weekly and yearly seasonality together**, plus a trend. It gets the same calendar information as SARIMA: England & Wales bank holidays (through Prophet's built-in holiday handling, using the names from data prep) and the Christmas-period flag as an extra input. Uncertainty sampling is switched off, since only point forecasts are needed and it makes each fit much faster.

In [7]:
import logging
from prophet import Prophet

# Prophet prints two log lines per fit; with 359 fits that floods the notebook
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)

# Prophet needs columns named exactly "ds" (date) and "y" (target)
prophet_df = pd.DataFrame({
    "ds": daily.index,
    "y": daily[TARGET].values,
    "is_christmas_period": daily["is_christmas_period"].values,
})

# Bank holidays in Prophet's format: one row per holiday date, with its name
holidays_df = (daily.loc[daily["is_holiday"] == 1, ["holiday_name"]]
                    .reset_index()
                    .rename(columns={"date": "ds", "holiday_name": "holiday"}))
print(len(holidays_df), "holiday dates,", holidays_df["holiday"].nunique(), "distinct holidays")


def fit_prophet(history):
    m = Prophet(
        yearly_seasonality=True,     # the cycle SARIMA can't model
        weekly_seasonality=True,
        daily_seasonality=False,     # data is daily, so there's no within-day pattern
        holidays=holidays_df,
        uncertainty_samples=0,       # skip interval sampling; much faster, and we only need point forecasts
    )
    m.add_regressor("is_christmas_period")   # same calendar information SARIMA gets
    m.fit(history)
    return m

48 holiday dates, 14 distinct holidays


/Users/jay/Documents/Project 5/uk-grid-demand-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 5b. Rolling forecasts
Prophet can't be updated with new data, so it's refitted from scratch at every origin (359 fits, under a minute in total). The actual demand is removed from the forecast input, so it can't leak into the predictions.

In [8]:
one_day = pd.Timedelta(days=1)
rows = []

t0 = time.time()
for i, origin in enumerate(origins, start=1):
    history = prophet_df[prophet_df["ds"] <= origin - one_day]
    future = prophet_df[prophet_df["ds"].between(origin, origin + pd.Timedelta(days=HORIZON - 1))]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = fit_prophet(history)
        forecast = m.predict(future.drop(columns="y"))   # drop y so the actual values can't leak in

    for h, (d, p) in enumerate(zip(forecast["ds"], forecast["yhat"]), start=1):
        rows.append({"model": "prophet", "origin": origin, "date": d,
                     "horizon": h, "y_true": y[d], "y_pred": p})

    if i == 1 or i % 50 == 0:
        elapsed = time.time() - t0
        print(f"  {i}/{len(origins)} done, {elapsed:.0f}s elapsed, ~{elapsed / i * (len(origins) - i) / 60:.0f} min left")

print(f"Rolling forecasts took {time.time() - t0:.0f}s")
prophet_preds = pd.DataFrame(rows)
prophet_scores = evaluate_and_save(prophet_preds, "prophet")

  1/359 done, 1s elapsed, ~5 min left
  50/359 done, 7s elapsed, ~1 min left
  100/359 done, 12s elapsed, ~1 min left
  150/359 done, 18s elapsed, ~0 min left
  200/359 done, 25s elapsed, ~0 min left
  250/359 done, 31s elapsed, ~0 min left
  300/359 done, 38s elapsed, ~0 min left
  350/359 done, 45s elapsed, ~0 min left
Rolling forecasts took 46s
=== prophet ===
  MAE   1,495.998
  RMSE  1,874.335
  MAPE  5.767
  MASE  0.858

  MAE by horizon (days ahead):
horizon
1    1430.0
2    1456.0
3    1477.0
4    1503.0
5    1520.0
6    1536.0
7    1550.0

  Saved 2513 rows to /Users/jay/Documents/Project 5/uk-grid-demand-forecasting/results/predictions/prophet.csv


## 6. Baseline results

Evaluation: 359 rolling 7-day forecasts across 2025 (a new forecast every day, each using only data up to the day before). Models trained on 2021–2024.

| Model | MAE (MW) | RMSE (MW) | MAPE | MASE |
|---|---|---|---|---|
| Seasonal naive | 1,822 | 2,392 | 6.89% | 1.045 |
| SARIMA (1,0,0)(2,1,1)[7] + holiday inputs | **1,493** | 1,934 | **5.76%** | **0.856** |
| Prophet + holidays | 1,496 | **1,874** | 5.77% | 0.858 |

**MAE by horizon (MW)**

| Days ahead | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| Seasonal naive | 1,844 | 1,852 | 1,837 | 1,823 | 1,812 | 1,800 | 1,786 |
| SARIMA | **973** | **1,247** | **1,376** | 1,541 | 1,666 | 1,784 | 1,865 |
| Prophet | 1,430 | 1,456 | 1,477 | **1,503** | **1,520** | **1,536** | **1,550** |

**Findings**
- **SARIMA and Prophet tie overall.** MAE is within 3 MW, and both are about 18% better than seasonal naive. Prophet's RMSE is about 3% lower, so it makes fewer large misses.
- **The winner depends on the horizon, with a crossover at day 4.** SARIMA is best for days 1–3 (day-1 MAE 973 MW vs Prophet's 1,430), because it uses the most recent observations directly. Prophet is best for days 4–7 and barely degrades (1,430 → 1,550 MW).
- **By day 7, SARIMA is no better than naive:** 1,865 MW vs 1,786 MW for "same day last week". Prophet is still 13% better than naive at day 7.
- **This partly confirms the EDA prediction.** Prophet's yearly seasonality helps at longer horizons, as expected, but SARIMA's short-range accuracy offsets it overall.
- **The two models fail in opposite ways,** so combining them (averaging, or using SARIMA for days 1–3 and Prophet for days 4–7) is a promising option for the final comparison.
- **Model choices:** auto-ARIMA chose `d = 0`, consistent with the ADF result. Weekly differencing (`D = 1`) was set from the EDA findings before any model was evaluated on 2025, so the test year played no part in choosing it.
- **Caveat:** 2025 was the most regular year in the data (lowest residual noise in the EDA), so errors in noisier years would probably be higher.